In [29]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import traceable, Client

# Initialize 
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o"

In [30]:
def get_temperature(city: str) -> str:
  """Get the current temperature for a city
  
  Args:
    city: The name of the city

  Returns:
    The current temperature for the city
  """
  temperatures = {
    "New York": "22°C",
    "London": "15°C",
    "Tokyo": "18°C"
  }
  return temperatures.get(city, "Unknown")

# tools = [
#     {
#         "type": "function",
#         "name": "get_temperature",
#         "description": "Get the temperature of a given city.",
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "city": {
#                     "type": "string",
#                     "description": "The name of the city",
#                 },
#             },
#             "required": ["city"],
#         },
#     },
# ]

In [31]:
# 1. Define the schema so the LLM knows the tool exists
tools = [{
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature in a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            },
            "required": ["city"]
        }
    }
}]

In [32]:
def call_openai(messages):
    """Wrapper to specifically trace the raw API call."""
    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
messages=[
    {"role": "system", "content": "You are a weather assistant. Use tools!"},
    {"role": "user", "content": "What's the temperature in New York?"}
]

# 2. The call (Fixing the typo 'temerature' just in case)
response = call_openai(messages=messages)
print(response)

ChatCompletion(id='chatcmpl-DY19NxfAhgIMXdNk4TVge8Eqb6FrT', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_m4Y9VZfygVchtrcnexheFIKP', function=Function(arguments='{"city":"New York"}', name='get_temperature'), type='function')]))], created=1776999945, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_50a29830e4', usage=CompletionUsage(completion_tokens=15, prompt_tokens=61, total_tokens=76, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [33]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-DY19NxfAhgIMXdNk4TVge8Eqb6FrT",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_m4Y9VZfygVchtrcnexheFIKP",
            "function": {
              "arguments": "{\"city\":\"New York\"}",
              "name": "get_temperature"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1776999945,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_50a29830e4",
  "usage": {
    "completion_tokens": 15,
    "prompt_tokens": 61,
    "total_tokens": 76,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0

In [34]:
# 3. CRITICAL: You must handle the response
assistant_msg = response.choices[0].message
print(assistant_msg)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_m4Y9VZfygVchtrcnexheFIKP', function=Function(arguments='{"city":"New York"}', name='get_temperature'), type='function')])


In [35]:
print(assistant_msg.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_m4Y9VZfygVchtrcnexheFIKP', function=Function(arguments='{"city":"New York"}', name='get_temperature'), type='function')]


In [36]:
for func in assistant_msg.tool_calls:
    print(func.function.name)
    print(json.dumps(func.function.arguments))

get_temperature
"{\"city\":\"New York\"}"


In [37]:
funcs = {"get_temperature": get_temperature}

In [38]:
# ... inside your if assistant_msg.tool_calls block ...

if assistant_msg.tool_calls:
    # --- STEP 1: FIX IS HERE ---
    # You MUST add the assistant's request to the history
    # This contains the 'tool_calls' that the 'tool' role responds to.
    messages.append(assistant_msg) 

    for tool_call in assistant_msg.tool_calls:
        function_name = tool_call.function.name
        # Use loads (plural) for the string
        args = json.loads(tool_call.function.arguments)
        
        print(f"  [Executing] {function_name} with {args}")
        
        result = funcs[function_name](**args)

        # --- STEP 2: ADD THE TOOL RESULT ---
        messages.append({
            "role": "tool", 
            "tool_call_id": tool_call.id, 
            "content": str(result)
        })

    # --- STEP 3: GET FINAL ANSWER ---
    # Now messages contains: [System, User, Assistant(tool_calls), Tool(result)]
    final_response = call_openai(messages)
    print(f"\nFinal Answer: {final_response.choices[0].message.content}")

  [Executing] get_temperature with {'city': 'New York'}

Final Answer: The current temperature in New York is 22°C.


### Multi Turn Tool calls

In [41]:
# @traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product.lower(), 0.0)

# @traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier.lower(), 0)
    return round(price * (1 - discount / 100), 2)

# --- Tool Schemas ---

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_product_price",
            "description": "Look up the price of a product in the catalog.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {"type": "string"},
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount",
            "description": "Apply a discount tier to a price.",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {"type": "number"},
                    "discount_tier": {"type": "string", "enum": ["bronze", "silver", "gold"]},
                },
                "required": ["price", "discount_tier"],
            },
        },
    }
]

MAX_ITERATIONS = 10

# --- Wrapped LLM Call ---

# @traceable(name="OpenAI Call", run_type="llm")
def call_openai(messages):
    """Wrapper to specifically trace the raw API call."""
    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

def run_agent(question: str):
    available_functions = {
        "get_product_price": get_product_price,
        "apply_discount": apply_discount,
    }
    messages = [
        {
            "role": "system", 
            "content": (
                "You are a shopping assistant. "
                "1. Use 'get_product_price' to find the base price of a product. "
                "2. Use 'apply_discount' to calculate the final price. "
                "3. If a user mentions a product like 'laptop', use that exact string in the tool. "
                "4. NEVER ask for more details if the product is likely in your catalog. "
                "5. ALWAYS call the tools before answering."
            )
        },
        {"role": "user", "content": question}
    ]

    print(f"Question: {question}\n" + "="*40)

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(F"\n ----- Iteration {iteration} ------ ")
        response = call_openai(messages=messages)
        ai_messages = response.choices[0].message
        messages.append(ai_messages)

        tool_calls = ai_messages.tool_calls
        if not tool_calls:
            print(f"\nFinal Answer: {ai_messages.content}")
            return ai_messages.content
        
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            args = tool_call.function.arguments
            print(f"  [Tool Selected] {tool_name}({args})")
            print(f"  [Tool selected] {tool_call} with args: {args}")
            tool_to_use = available_functions[tool_name]
            if not tool_to_use:
                raise ValueError(f"Tool {tool_name} not found")
            observation = tool_to_use(**json.loads(args))
            print(f"Tool result: {observation}")

            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_name,
                "content": str(observation),
            })

    print("ERROR: Max iterations reached without a final answer")
    return None

print("Langchain agent with Binding tools working already")
print()
result = run_agent("What is the price of laptop after applying a gold discount?")

Langchain agent with Binding tools working already

Question: What is the price of laptop after applying a gold discount?

 ----- Iteration 1 ------ 
  [Tool Selected] get_product_price({"product":"laptop"})
  [Tool selected] ChatCompletionMessageFunctionToolCall(id='call_CMPFyi9yTf4z6HfmpHIVBtrD', function=Function(arguments='{"product":"laptop"}', name='get_product_price'), type='function') with args: {"product":"laptop"}
    >> Executing get_product_price(product='laptop')
Tool result: 1299.99

 ----- Iteration 2 ------ 
  [Tool Selected] apply_discount({"price":1299.99,"discount_tier":"gold"})
  [Tool selected] ChatCompletionMessageFunctionToolCall(id='call_kYpBBVHkaQO641ccv8xMI37Z', function=Function(arguments='{"price":1299.99,"discount_tier":"gold"}', name='apply_discount'), type='function') with args: {"price":1299.99,"discount_tier":"gold"}
    >> Executing apply_discount(price=1299.99, discount_tier='gold')
Tool result: 1000.99

 ----- Iteration 3 ------ 

Final Answer: The p

In [42]:
# @traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product.lower(), 0.0)

# @traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier.lower(), 0)
    return round(price * (1 - discount / 100), 2)

# --- Tool Schemas ---

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_product_price",
            "description": "Look up the price of a product in the catalog.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {"type": "string"},
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount",
            "description": "Apply a discount tier to a price.",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {"type": "number"},
                    "discount_tier": {"type": "string", "enum": ["bronze", "silver", "gold"]},
                },
                "required": ["price", "discount_tier"],
            },
        },
    }
]

MAX_ITERATIONS = 10

# --- Wrapped LLM Call ---

# @traceable(name="OpenAI Call", run_type="llm")
def call_openai(messages):
    """Wrapper to specifically trace the raw API call."""
    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

def run_agent(question: str):
    available_functions = {
        "get_product_price": get_product_price,
        "apply_discount": apply_discount,
    }
    messages = [
        {
            "role": "system", 
            "content": (
                "You are a shopping assistant. "
                "1. Use 'get_product_price' to find the base price of a product. "
                "2. Use 'apply_discount' to calculate the final price. "
                "3. If a user mentions a product like 'laptop', use that exact string in the tool. "
                "4. NEVER ask for more details if the product is likely in your catalog. "
                "5. ALWAYS call the tools before answering."
            )
        },
        {"role": "user", "content": question}
    ]

    print(f"Question: {question}\n" + "="*40)

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(F"\n ----- Iteration {iteration} ------ ")
        response = call_openai(messages=messages)
        ai_messages = response.choices[0].message
        messages.append(ai_messages)

        tool_calls = ai_messages.tool_calls
        if not tool_calls:
            print(f"\nFinal Answer: {ai_messages.content}")
            return ai_messages.content
        
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            args = tool_call.function.arguments
            print(f"  [Tool Selected] {tool_name}({args})")
            print(f"  [Tool selected] {tool_call} with args: {args}")
            tool_to_use = available_functions[tool_name]
            if not tool_to_use:
                raise ValueError(f"Tool {tool_name} not found")
            observation = tool_to_use(**json.loads(args))
            print(f"Tool result: {observation}")

            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_name,
                "content": str(observation),
            })

    print("ERROR: Max iterations reached without a final answer")
    return None

print("Langchain agent with Binding tools working already")
print()
result = run_agent("What is the price of laptop after applying a silver discount?")

Langchain agent with Binding tools working already

Question: What is the price of laptop after applying a silver discount?

 ----- Iteration 1 ------ 
  [Tool Selected] get_product_price({"product":"laptop"})
  [Tool selected] ChatCompletionMessageFunctionToolCall(id='call_G3H5hcWyHGTOuxXMYozCWdmT', function=Function(arguments='{"product":"laptop"}', name='get_product_price'), type='function') with args: {"product":"laptop"}
    >> Executing get_product_price(product='laptop')
Tool result: 1299.99

 ----- Iteration 2 ------ 
  [Tool Selected] apply_discount({"price":1299.99,"discount_tier":"silver"})
  [Tool selected] ChatCompletionMessageFunctionToolCall(id='call_RqufucaH9ANvc46nTx4C3NaG', function=Function(arguments='{"price":1299.99,"discount_tier":"silver"}', name='apply_discount'), type='function') with args: {"price":1299.99,"discount_tier":"silver"}
    >> Executing apply_discount(price=1299.99, discount_tier='silver')
Tool result: 1143.99

 ----- Iteration 3 ------ 

Final Ans

## Using ResponseAPI

In [45]:
def get_temperature(city: str) -> str:
  """Get the current temperature for a city
  
  Args:
    city: The name of the city

  Returns:
    The current temperature for the city
  """
  temperatures = {
    "New York": "22°C",
    "London": "15°C",
    "Tokyo": "18°C"
  }
  return temperatures.get(city, "Unknown")

tools = [
    {
        "type": "function",
        "name": "get_temperature",
        "description": "Get the temperature of a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city",
                },
            },
            "required": ["city"],
        },
    },
]

from openai.types.responses import Response

def call_openai_responses(messages: list) -> Response:
    """Wrapper using the new Responses API primitive."""
    return client.responses.create(
        model="gpt-4o",
        input=messages,  # Note: Responses API uses 'input' instead of 'messages'
        tools=tools
    )

messages = [
        {
            "role": "system", "content": "You are a weather assistant. Use tools!"
        },
        {"role": "user", "content": "Whats the current temperature of New York"}
    ]

response = call_openai_responses(messages=messages)
print(response)
  

Response(id='resp_097a775d6fdfbd4d0069eae4cde108819c80c0c11dde382754', created_at=1777001677.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-2024-08-06', object='response', output=[ResponseFunctionToolCall(arguments='{"city":"New York"}', call_id='call_M4b3Eh3gIlcstfeGLZRrX6XP', name='get_temperature', type='function_call', id='fc_097a775d6fdfbd4d0069eae4d05504819c9012bed10340ee2f', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_temperature', parameters={'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'The name of the city'}}, 'required': ['city'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Get the temperature of a given city.')], top_p=1.0, background=False, completed_at=1777001680.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, 

In [46]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_097a775d6fdfbd4d0069eae4cde108819c80c0c11dde382754",
  "created_at": 1777001677.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-2024-08-06",
  "object": "response",
  "output": [
    {
      "arguments": "{\"city\":\"New York\"}",
      "call_id": "call_M4b3Eh3gIlcstfeGLZRrX6XP",
      "name": "get_temperature",
      "type": "function_call",
      "id": "fc_097a775d6fdfbd4d0069eae4d05504819c9012bed10340ee2f",
      "namespace": null,
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_temperature",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The name of the city"
          }
        },
        "required": [
          "city"
        ],
        "additionalProperties": false
      },
      "strict": true,

In [47]:
print(response.tools)

[FunctionTool(name='get_temperature', parameters={'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'The name of the city'}}, 'required': ['city'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Get the temperature of a given city.')]


In [48]:
print(type(response.tools))

<class 'list'>


In [51]:
for tool in response.tools:
    print(tool)

FunctionTool(name='get_temperature', parameters={'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'The name of the city'}}, 'required': ['city'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Get the temperature of a given city.')


In [52]:
for tool in response.output:
    print(tool.name)
    print(json.loads(tool.arguments))

get_temperature
{'city': 'New York'}


In [77]:
# @traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


# @traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.
    Available tiers: bronze, silver, gold."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)

tools = [
    {
        "type": "function",
        "name": "get_product_price",
        "description": "Look up the price of a product in the catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "product": {
                   "type": "string",
                   "description": "The product name, e.g. 'laptop', 'headphones', 'keyboard'"
                },
            },
            "required": ["product"],
        },
    },
    {
        "type": "function",
        "name": "apply_discount",
        "description": "Apply a discount tier to a price.",
        "parameters": {
        "type": "object",
        "properties": {
            "price": {"type": "number"},
                "discount_tier": {
                   "type": "string", 
                   "enum": ["bronze", "silver", "gold"],
                   "description": "The discount tier: 'bronze', 'silver', or 'gold'"
                },
            },
            "required": ["price", "discount_tier"],
        },
    }
]

tools = [
    {
        "type": "function",
        "name": "get_product_price",  # Name is top-level here
        "description": "Look up the price of a product in the catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "product": {"type": "string"}
            },
            "required": ["product"]
        }
    },
    {
        "type": "function",
        "name": "apply_discount",  # Name is top-level here
        "description": "Apply a discount tier to a price.",
        "parameters": {
            "type": "object",
            "properties": {
                "price": {"type": "number"},
                "discount_tier": {"type": "string", "enum": ["bronze", "silver", "gold"]}
            },
            "required": ["price", "discount_tier"]
        }
    }
]

from openai.types.responses import Response

def call_openai_responses(messages: list) -> Response:
    """Wrapper using the new Responses API primitive."""
    return client.responses.create(
        model="gpt-4o",
        input=messages,  # Note: Responses API uses 'input' instead of 'messages'
        tools=tools
    )
messages = [
        {
            "role": "system", 
            "content": (
                "You are a shopping assistant. "
                "1. Use 'get_product_price' to find the base price of a product. "
                "2. Use 'apply_discount' to calculate the final price. "
                "3. If a user mentions a product like 'laptop', use that exact string in the tool. "
                "4. NEVER ask for more details if the product is likely in your catalog. "
                "5. ALWAYS call the tools before answering."
                "6. Dont ever guess a number for price use exactly whats provided for each product, and the call the apply_function immediately after the actual price is available"
            )
        },
        {"role": "user", "content": "What's the price of laptop after applying gold discount"}
    ]

response = call_openai_responses(messages=messages)
print(response)
  

Response(id='resp_0f4e39b249bf57b60069eaf366475481978d12c86922e2b2d7', created_at=1777005414.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-2024-08-06', object='response', output=[ResponseFunctionToolCall(arguments='{"product":"laptop"}', call_id='call_Ykv8Udxc7jLum8x1VxOrCKPz', name='get_product_price', type='function_call', id='fc_0f4e39b249bf57b60069eaf3674b6c8197930ce9528df34df3', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_product_price', parameters={'type': 'object', 'properties': {'product': {'type': 'string'}}, 'required': ['product'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Look up the price of a product in the catalog.'), FunctionTool(name='apply_discount', parameters={'type': 'object', 'properties': {'price': {'type': 'number'}, 'discount_tier': {'type': 'string', 'enum': ['bronze', 'silver', 'gol

In [78]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0f4e39b249bf57b60069eaf366475481978d12c86922e2b2d7",
  "created_at": 1777005414.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-2024-08-06",
  "object": "response",
  "output": [
    {
      "arguments": "{\"product\":\"laptop\"}",
      "call_id": "call_Ykv8Udxc7jLum8x1VxOrCKPz",
      "name": "get_product_price",
      "type": "function_call",
      "id": "fc_0f4e39b249bf57b60069eaf3674b6c8197930ce9528df34df3",
      "namespace": null,
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_product_price",
      "parameters": {
        "type": "object",
        "properties": {
          "product": {
            "type": "string"
          }
        },
        "required": [
          "product"
        ],
        "additionalProperties": false
      },
      "strict": true,
      "type": "function",
      "defer_

In [79]:
for tool in response.tools:
    print(tool)

FunctionTool(name='get_product_price', parameters={'type': 'object', 'properties': {'product': {'type': 'string'}}, 'required': ['product'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Look up the price of a product in the catalog.')
FunctionTool(name='apply_discount', parameters={'type': 'object', 'properties': {'price': {'type': 'number'}, 'discount_tier': {'type': 'string', 'enum': ['bronze', 'silver', 'gold']}}, 'required': ['price', 'discount_tier'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Apply a discount tier to a price.')


In [80]:
for tool in response.output:
    print(tool.name)
    print(json.loads(tool.arguments))

get_product_price
{'product': 'laptop'}


In [66]:
for tool in response.output:
    print(tool)

ResponseFunctionToolCall(arguments='{"product":"laptop"}', call_id='call_x6EDvLF8CHRpk6llY32wBpKs', name='get_product_price', type='function_call', id='fc_008b7dc57fdb19230069eaee8a26448196bb033b76d8c490e7', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"price":0,"discount_tier":"gold"}', call_id='call_2xXZ4mU8CX81g6hBcCVSDKFJ', name='apply_discount', type='function_call', id='fc_008b7dc57fdb19230069eaee8a265481969a7ca02db99f46b2', namespace=None, status='completed')


In [98]:
from openai import OpenAI
import json

client = OpenAI()

# 1. Tools Schema (Flattened for Responses API)
tools = [
    {
        "type": "function",
        "name": "get_product_price",
        "description": "Look up the price of a product in the catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "product": {"type": "string"}
            },
            "required": ["product"]
        }
    },
    {
        "type": "function",
        "name": "apply_discount",
        "description": "Apply a discount tier to a price.",
        "parameters": {
            "type": "object",
            "properties": {
                "price": {"type": "number"},
                "discount_tier": {"type": "string", "enum": ["bronze", "silver", "gold"]}
            },
            "required": ["price", "discount_tier"]
        }
    }
]

# 2. Optimized Wrapper
def call_openai_responses(input_data: list):
    return client.responses.create(
        model="gpt-4o",
        input=input_data,
        tools=tools,
        temperature=0  # CRITICAL: Keeps numbers exact
    )

# 3. Initial Execution
messages = [
    {
        "role": "system", 
        "content": "Shopping assistant. Step 1: Get price. Step 2: Apply discount with EXACT price. No math."
    },
    {"role": "user", "content": "What's the price of laptop after applying gold discount?"}
]

response = call_openai_responses(input_data=messages)

# How to read the output:
for item in response.output:
    if item.type == "function_call":
        print(f"Model wants to call {item.name} with {item.arguments}")
    elif item.type == "text":
        print(f"Model says: {item.text.content}")

Model wants to call get_product_price with {"product":"laptop"}


In [101]:
def run_agent(question: str):
    available_functions = {
        "get_product_price": get_product_price,
        "apply_discount": apply_discount,
    }
    messages = [
        {"role": "system", "content": "You are a shopping assistant. Use EXACT values from tools."},
        {"role": "user", "content": question}
    ]

    # Limit iterations so we don't get stuck in an infinite loop
    for iteration in range(10):
        response = client.responses.create(
            model="gpt-4o",
            input=messages,
            tools=tools,
            temperature=0
        )

        # IMPORTANT: Add the model's thoughts/calls to history
        messages += response.output

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            # No more tools? This is the final answer!
            return response.output_text

        # If there are tool calls, we must process them and continue the loop
        for tool in tool_calls:
            # 1. Get the function
            func = available_functions[tool.name]
            
            # 2. Run it
            result = func(**json.loads(tool.arguments))
            print(f"Iteration {iteration}: {tool.name} -> {result}")

            # 3. Add the result back to messages as a 'function_call_output' item
            messages.append({
                "type": "function_call_output",
                "call_id": tool.call_id,
                "output": str(result)
            })

        # The loop starts over, calling the API with the new data

In [102]:
print("Langchain agent with Binding tools working already")
print()
result = run_agent("What is the price of laptop after applying a gold discount?")

Langchain agent with Binding tools working already

    >> Executing get_product_price(product='laptop')
Iteration 0: get_product_price -> 1299.99
    >> Executing apply_discount(price=1299.99, discount_tier='gold')
Iteration 1: apply_discount -> 1000.99


In [89]:
# @traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product.lower(), 0.0)

# @traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier.lower(), 0)
    return round(price * (1 - discount / 100), 2)

# --- Tool Schemas ---

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_product_price",
            "description": "Look up the price of a product in the catalog.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {"type": "string"},
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount",
            "description": "Apply a discount tier to a price.",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {"type": "number"},
                    "discount_tier": {"type": "string", "enum": ["bronze", "silver", "gold"]},
                },
                "required": ["price", "discount_tier"],
            },
        },
    }
]

MAX_ITERATIONS = 10

# --- Wrapped LLM Call ---

# @traceable(name="OpenAI Call", run_type="llm")
def call_openai(messages):
    """Wrapper to specifically trace the raw API call."""
    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

def run_agent(question: str):
    available_functions = {
        "get_product_price": get_product_price,
        "apply_discount": apply_discount,
    }
    messages = [
        {
            "role": "system", 
            "content": (
                "You are a shopping assistant. "
                "1. Use 'get_product_price' to find the base price of a product. "
                "2. Use 'apply_discount' to calculate the final price. "
                "3. If a user mentions a product like 'laptop', use that exact string in the tool. "
                "4. NEVER ask for more details if the product is likely in your catalog. "
                "5. ALWAYS call the tools before answering."
            )
        },
        {"role": "user", "content": question}
    ]

    print(f"Question: {question}\n" + "="*40)

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(F"\n ----- Iteration {iteration} ------ ")
        response = call_openai(messages=messages)
        ai_messages = response.choices[0].message
        messages.append(ai_messages)

        tool_calls = ai_messages.tool_calls
        if not tool_calls:
            print(f"\nFinal Answer: {ai_messages.content}")
            return ai_messages.content
        
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            args = tool_call.function.arguments
            print(f"  [Tool Selected] {tool_name}({args})")
            print(f"  [Tool selected] {tool_call} with args: {args}")
            tool_to_use = available_functions[tool_name]
            if not tool_to_use:
                raise ValueError(f"Tool {tool_name} not found")
            observation = tool_to_use(**json.loads(args))
            print(f"Tool result: {observation}")

            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_name,
                "content": str(observation),
            })

    print("ERROR: Max iterations reached without a final answer")
    return None

print("Langchain agent with Binding tools working already")
print()
result = run_agent("What is the price of laptop after applying a silver discount?")

Langchain agent with Binding tools working already

Question: What is the price of laptop after applying a silver discount?

 ----- Iteration 1 ------ 


  [Tool Selected] get_product_price({"product":"laptop"})
  [Tool selected] ChatCompletionMessageFunctionToolCall(id='call_hP2wghL8pwQ1FE2E0DJDAWdf', function=Function(arguments='{"product":"laptop"}', name='get_product_price'), type='function') with args: {"product":"laptop"}
    >> Executing get_product_price(product='laptop')
Tool result: 1299.99

 ----- Iteration 2 ------ 
  [Tool Selected] apply_discount({"price":1299.99,"discount_tier":"silver"})
  [Tool selected] ChatCompletionMessageFunctionToolCall(id='call_Q7wD2pxdhB9qt1ZLRCvMUwLD', function=Function(arguments='{"price":1299.99,"discount_tier":"silver"}', name='apply_discount'), type='function') with args: {"price":1299.99,"discount_tier":"silver"}
    >> Executing apply_discount(price=1299.99, discount_tier='silver')
Tool result: 1143.99

 ----- Iteration 3 ------ 

Final Answer: The price of the laptop after applying a silver discount is $1,143.99.
